<a href="https://colab.research.google.com/github/anabelpoulard13-boop/ensight_concentration_analysis/blob/main/concentration_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Concentration Analysis

The following code performs a concentration analysis on a `.csv` file with the absorbance measurements from EnSight plate reader.

It takes the raw `.csv` and, using key information provided at the start, estimates the cell growth function and applies it to all wells in the plate. It then cleans the data to keep only relevant growth and outputs an Excel table along with a list of wells showing significant growth. As a bonus, it also generates a heatmap to help visualize the relevant wells.

## Variables
The variables blank_well, sample_well, and sample_concentration must be defined before running the code. They tell the program which wells to use for the slope calculation.

- `blank_well`, is a well with no concentration but still has an absorbance value therfore it has always a value of 0.

- `sample_well`, is a well with a known sample

- `sample_concentration`, is the concentration of that sample.

In [ ]:
blank_well = 'O6' # Well with no sample 0ng/ul
sample_well = 'A6' # Well with sample
sample_concentration = 100 # Sample concentration in ng/ul

### Data frame
This part of the code takes the raw data from the Ensight machine and converts it into a data frame so Python can extract and work only with the relevant data.

In [ ]:
import pandas as pd
import string
import glob
import matplotlib.pyplot as plt
import numpy as np
import os

files_path = []
files_path = glob.glob('*.csv')
file_path = files_path[0]
print(file_path)

plates_sizes = {                  # This is a disctionary of possible plate
    6 : {'rows': 2, 'cols': 3},   # size allowing the user to have different size of plate
    12 : {'rows': 3, 'cols': 4},
    24 : {'rows': 4, 'cols': 6},
    96 : {'rows': 8, 'cols': 12},
    384 : {'rows': 16, 'cols': 24}
}

df = pd.read_csv(file_path, skiprows = 9)
print(df.head())

num_cols = len(df.columns) - 2

detected_plate = None
for row, col in plates_sizes.items():
    if col['cols'] == num_cols:
        detected_plate = row
        break
if detected_plate:
    read_this = plates_sizes[detected_plate]['rows']
    print(f'✓ Detected: {detected_plate} well plate — reading {read_this} rows')
    df = pd.read_csv(file_path, skiprows=9, nrows= read_this)
else:
    print(f'✗ Unknown plate size — {num_cols} columns not in list')

table_data = df.iloc[0:read_this, 1:num_cols + 1]
row_labels = [string.ascii_uppercase[i] for i in range(len(table_data))]
table_data.insert(0, 'Row_Label', row_labels)

display(table_data)

### Finding the Variable
This code looks through the data frame to find the `blank_well` and the sample_well. Once it finds them, it assigns a value of 0 ng/µL to the blank_well and `the sample_concentration` to the `sample_well`.

In [ ]:
def get_well_value(df, well):
    row_label = well[0]
    col_number = int(well[1:])
    row = df[df['Row_Label'] == row_label]
    return row.iloc[0, col_number]

zero_point_value = get_well_value(table_data, blank_well)
zero_point_concentration = 0

print(f"0ng/ul standard: Measured Value = {zero_point_value}, Concentration = {zero_point_concentration}ng/ul")

point_value = get_well_value(table_data, sample_well)
point_concentration = sample_concentration

print(f"{sample_concentration}ng/ul standard: Measured Value = {point_value}, Concentration = {point_concentration}ng/ul")


### Growth calculation
This part uses the values defined earlier to calculate the slope and the y-intercept. It then defines a formula that can be applied to the rest of the cells in the data frame.

Here, the y-axis represents absorbance and the x-axis represents concentration.

In [ ]:
m = (point_value - zero_point_value) / (point_concentration - zero_point_concentration)
b = zero_point_value - (m * zero_point_concentration)

print(f"Calculated slope (m): {m:.2f}")
print(f"Calculated y-intercept (b): {b:.2f}")

def calculate_concentration(measured_value):
  concentration = (2 * ((measured_value - b)/ m))
  return concentration.round(2)

### Calculation Application
This part of the code applies the formula defined earlier to all the cells in the data frame.

It then generates a linear plot showing the growth approximation based on the two given points.

In [ ]:
for col in [col for col in table_data.columns if col != 'Row_Label']:
    table_data[col] = table_data[col].apply(calculate_concentration)

print("Here is the data with calculated concentrations:")
display(table_data)

x_values = [zero_point_concentration, point_concentration]
y_values = [zero_point_value, point_value]

plt.figure(figsize=(8, 5))
plt.plot(x_values, y_values, color='blue', linestyle='-', marker='o')
plt.xlim(0, max(x_values) * 1.1)
plt.ylim(0, max(y_values) * 1.1)
plt.title('Calibration Curve')
plt.ylabel('Measured Fluorescence Value')
plt.xlabel('Concentration (ng/ul)')
plt.grid(True)
plt.show()

### Clean up
This part of the code removes all negative values from the data frame, since they are not useful for analysis.

In [ ]:
table_data1 = table_data.where(table_data.select_dtypes(include='number') >= 0)
new_table = table_data1[table_data1 >= 0].fillna('')
new_table['Row_Label'] = table_data['Row_Label']
display(new_table)

### Heatmap
This code generates a heatmap of the relevant concentrations. It uses a color scale, with lighter shades for lower concentration and darker shades for higher concentrations.

In [ ]:
# Separate Row_Label and convert the rest to numeric
row_labels = new_table['Row_Label'].tolist()
col_labels = [col for col in new_table.columns if col != 'Row_Label']

# Convert to float matrix (empty strings become NaN)
harvest = new_table[col_labels].replace('', np.nan).astype(float).values

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(harvest, aspect='auto', cmap='YlOrRd')

# Set ticks and labels
ax.set_xticks(range(len(col_labels)), labels=col_labels)
ax.set_yticks(range(len(row_labels)), labels=row_labels)

# Add colorbar
plt.colorbar(im, ax=ax, label='Concentration (ng/ul)')

# Loop over data and create text annotations
for i in range(len(row_labels)):
    for j in range(len(col_labels)):
        value = harvest[i, j]
        if not np.isnan(value):
          text_color = "white" if value > np.nanmax(harvest) / 2 else "black"
          text = ax.text(j, i, f"{value:.2f}",
                          ha="center", va="center", color= text_color, fontsize=10)

ax.set_title("Concentration Heatmap (ng/ul)")
fig.tight_layout()
plt.show()

### Excel table
This code lets the user export the cleaned data frame as an Excel file for further analysis specific to the experiment. The file is available in the Colab document under the name “result.”

In [ ]:
if not os.path.exists('result'):
    os.mkdir('result')
data_table = new_table.copy()
data_table['Row_Label'] = new_table['Row_Label']
data_table.to_excel('result/data_table.xlsx', index=False)

### CSV List
This final part of the code generates a CSV file listing all the well positions with their corresponding concentrations. The csv list is available in the Colab document in the file named “result.” To view the full list, open it in the Colab document.

In [ ]:
rows = []
for i, row in new_table.iterrows():
    row_label = row['Row_Label']
    for col in [col for col in new_table.columns if col != 'Row_Label']:
        value = row[col]
        if pd.notna(value) and value != '':
            rows.append({'Well': f"{row_label}{col}", 'Concentration_ng_ul': value})

csv_df = pd.DataFrame(rows)
csv_df.to_csv('result/well_concentrations.csv', index=False)
print(f"Saved {len(csv_df)} wells to CSV!")
display(csv_df)